# Training the expert-feature experiments on Colab

Colab gives you a free GPU. This project trains in about 2.5 hours on the
laptop's CPU and a few minutes on a Colab T4 -- that difference is the whole
reason to use it.

Colab is a throwaway machine. Nothing here touches your laptop, and
**everything is deleted when the session ends**, so anything worth keeping
must be written back to Drive before you close the tab. That is the one rule
that catches people out.

Run ONE experiment per session, so you can tell which change did what.


## 1. Turn on the GPU

**Runtime -> Change runtime type -> T4 GPU -> Save.** Do this first; changing
it later restarts the session and you lose everything above it.

If the cell prints `CUDA: False` you are on CPU and training will take hours.


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'CPU ONLY -- go back and change the runtime type')


## 2. Get the code

Colab clones from GitHub. It cannot see anything that exists only on your
laptop, so **push the branch first**:

```
git push origin eileen-omni-ui
```

Without that, the `cumulant_features` and `if_features` flags will not exist
here and the training cell fails.


In [ ]:
%cd /content
!rm -rf sedicAI_NEXA
!git clone -q -b eileen-omni-ui https://github.com/eavan127/sedicAI_NEXA.git
%cd /content/sedicAI_NEXA
!git log --oneline -3
!pip install -q pyyaml h5py


## 3. Get the data in

Training needs three arrays that are NOT in git -- `X.npy` alone is 330 MB.
They are already in your Drive, so mount it and copy. Do not upload through
the browser: 330 MB over a browser tab is slow and drops often.

Mounting shows a permissions prompt; it grants access for this session only.

**Set `DRIVE_DATA` to the folder in your Drive holding the three .npy files.**
Use the folder icon in the left sidebar to find the path after mounting.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import shutil, pathlib

# EDIT THIS to the Drive folder containing X.npy, y.npy, snr_labels.npy
DRIVE_DATA = '/content/drive/MyDrive/sedicAI_NEXA/data/processed'

dest = pathlib.Path('/content/sedicAI_NEXA/data/processed')
dest.mkdir(parents=True, exist_ok=True)
for name in ('X.npy', 'y.npy', 'snr_labels.npy'):
    src = pathlib.Path(DRIVE_DATA) / name
    assert src.exists(), f'not found: {src} -- fix DRIVE_DATA above'
    shutil.copy(src, dest / name)
    print(f'{name:16s} {(dest / name).stat().st_size / 1e6:8.1f} MB')


In [ ]:
# Check the shapes before spending GPU time on them.
import numpy as np
X = np.load('data/processed/X.npy', mmap_mode='r')
y = np.load('data/processed/y.npy', mmap_mode='r')
snr = np.load('data/processed/snr_labels.npy', mmap_mode='r')
print('X  ', X.shape, X.dtype, ' expect (80400, 2, 512) float32')
print('y  ', y.shape, ' expect (80400, 8) multi-hot')
print('snr', snr.shape, ' bins', sorted(set(snr.tolist())))


## 4. Choose the experiment

One flag per session. Both at once is faster in wall-clock terms but tells
you nothing about attribution if it works.

| flag | targets | baseline to beat |
|---|---|---|
| `cumulant_features` | 16QAM vs 64QAM is a coin flip | combined 51.9% |
| `if_features` | radar and FHSS confuse each other | 50.7% of radar FPs are FHSS |

Both are off by default so the shipped checkpoints keep loading. Switching one
on changes the model's input width, so a checkpoint trained with it can only
be loaded with the same flag on -- which is why the probe below takes a flag.


In [ ]:
EXPERIMENT = 'cumulant_features'   # or 'if_features'
SEED = 2000                        # member 0's seed, matching the pinned baseline

import sys; sys.path.insert(0, '/content/sedicAI_NEXA')
from src.config import CFG
CFG.setdefault('model', {})[EXPERIMENT] = True
print(EXPERIMENT, '->', CFG['model'][EXPERIMENT])
print('epochs', CFG['training']['epochs'], ' batch', CFG['training']['batch_size'])


## 5. Train

Roughly 5-15 minutes on a T4, against 2.5 hours on the laptop.

Per-epoch train and validation loss print as it goes and are written to a
JSON beside the checkpoint, so you can see afterwards whether it overfitted.
The checkpoint is saved every time validation improves, so a disconnect at
epoch 25 costs you the last few epochs rather than the whole run.

**Do not close the tab** -- Colab kills idle sessions.


In [ ]:
import torch, pathlib
from src.train import load_data, stratified_split
from scripts.train_ensemble import train_one

X, y, snr_labels = load_data()
d = CFG['dataset']
tr, va, _ = stratified_split(y, snr_labels, d['val_frac'], d['test_frac'], d['seed'])
print(f'train {len(tr)}  val {len(va)}')

out = pathlib.Path(f'/content/sedicAI_NEXA/results/expert_{EXPERIMENT}.pt')
hist = pathlib.Path(f'/content/sedicAI_NEXA/results/expert_{EXPERIMENT}_history.json')
out.parent.mkdir(parents=True, exist_ok=True)

model = train_one(X, y, snr_labels, tr, va, seed=SEED,
                  history_path=hist, ckpt_path=out)
torch.save(model.state_dict(), out)
print('saved', out)


## 6. Save to Drive BEFORE closing the tab

The step people forget. `/content` is wiped when the session ends.


In [ ]:
import shutil, pathlib
DRIVE_OUT = '/content/drive/MyDrive/sedicAI_NEXA_experiments'
pathlib.Path(DRIVE_OUT).mkdir(parents=True, exist_ok=True)
for f in (out, hist):
    shutil.copy(f, pathlib.Path(DRIVE_OUT) / f.name)
    print('copied to Drive:', f.name)


## 7. Did it work?

The probe reports both weaknesses and the judged-class recall together.
Judged recall is there deliberately: these classes share a decision boundary,
and a change that cleans up the confusion by giving up the 80% gate is not a
fix.

This runs a SINGLE model while the pinned baseline is the 5-model ensemble, so
expect judged recall a couple of points lower for that reason alone. What
matters is whether the QAM and confusion numbers move.


In [ ]:
flag = '--cumulant-features' if EXPERIMENT == 'cumulant_features' else '--if-features'
!python scripts/probe_expert_features.py --checkpoint {out} {flag} --out results/probe_{EXPERIMENT}.json


In [ ]:
# The pinned baseline, for side-by-side reading.
!cat docs/experiments/expert_features_baseline.json


In [ ]:
# Training curve -- did it overfit?
import json
h = json.load(open(hist))
print(f"best epoch {h['best_epoch']} of {len(h['epochs'])}   best val {h['best_val_loss']:.4f}")
for e in h['epochs']:
    mark = '  <- best' if e['improved'] else ''
    print(f"  {e['epoch']:2d}  train {e['train_loss']:.4f}  val {e['val_loss']:.4f}{mark}")
